In [1]:
pip install gensim scikit-learn seaborn matplotlib fasttext


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from gensim.models.fasttext import load_facebook_model
from gensim.models.callbacks import CallbackAny2Vec
import numpy as np
import pandas as pd
import os
import json
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_distances
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from gensim.models import FastText

In [3]:
# !curl -O https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.bin.gz

In [4]:
# !mkdir -p "Fasttext/Sin"
# !gunzip -c cc.si.300.bin.gz > "Fasttext/Sin/cc.si.300.bin"

In [5]:
pd.set_option('display.max_columns', None)

In [6]:
with open("../Data/century_tokens.json", "r", encoding="utf-8") as f:
    century_tokens_loaded = json.load(f)

In [7]:
century_tokens_loaded.keys()

dict_keys(['5', '12', '13', '14', '15', '16', '18', '19', '20'])

In [8]:
century_5th = century_tokens_loaded['5']
century_12th = century_tokens_loaded['12']
century_13th = century_tokens_loaded['13']
century_14th = century_tokens_loaded['14']
century_15th = century_tokens_loaded['15']
century_16th = century_tokens_loaded['16']
century_18th = century_tokens_loaded['18']
century_19th = century_tokens_loaded['19']
century_20th = century_tokens_loaded['20']

In [9]:
century_5th

[['කාම',
  'ලොක',
  'රූප',
  'ලොක',
  'අරූපලෝක',
  'යන්',
  'ලොකත්රයට',
  'ශ්රෙෂ්ඨ',
  'වූ',
  'මුනීන්ද්ර',
  'නම්',
  'වූ',
  'සම්යක්',
  'සම්බුද්ධයන්',
  'වහන්සේගේ',
  'ශ්රී',
  'පාද',
  'පද්මය',
  'භය',
  'ලොහ',
  'කුලාචාරයෙන්',
  'විනා',
  'සකසා',
  'වැඳ',
  'සකල',
  'ශාස්ත්රයෙහි',
  'පාරප්රාප්ත',
  'පණ්ඩිත',
  'වූ',
  'භෛෂජ්යාර්ථයෙහි',
  'දක්ෂ',
  'වූ',
  'වෙදුන්',
  'විසින්',
  'උතුම්',
  'වූ',
  'රස',
  'සංහිතාදි',
  'තන්ත්ර',
  'ශාස්ත්රයෙහි',
  'වුත',
  'කියන',
  'ලද',
  'විවිධං',
  'නානාප්රකාර',
  'වූ',
  'සාරාර්ථය',
  'උදුරා',
  'ගෙණ',
  'සකල',
  'සත්වයන්ගේ',
  'රොග',
  'ව්යපසමන',
  'සංඛ්යාත',
  'සංග්රහව',
  'පිණිස',
  'මේ',
  'සාරාර්ථ',
  'සංග්රහ',
  'නම්',
  'ප්රකරණය',
  'සර්ථාවබොධය',
  'සඳහා',
  'කියන්නෙමි'],
 ['යනාදී',
  'අනෙකවිධ',
  'මානයෙන්',
  'දැක්වූ',
  'මේ',
  'වෛද්ය',
  'ශාස්ත්රය',
  'ප්රථම',
  'කල්පයෙහි',
  'ලෝකාර්ථය',
  'කරන්නා',
  'වූ',
  'මහ',
  'බ්රහ්මයා',
  'දක්ෂ',
  'පුජාපති',
  'හට',
  'කීහ'],
 ['දක්ෂ', 'ප්රජාපති', 'අග', 'විනීන්', 'දෙදෙනා', 'හට', 'කීහ'],
 [

In [10]:
print(f"""Number of sentences in 5th century: {len(century_5th)}\n
      Number of sentences in 12th century: {len(century_12th)}\n
      Number of sentences in 13th century: {len(century_13th)}\n
      Number of sentences in 14th century: {len(century_14th)}\n
      Number of sentences in 15th century: {len(century_15th)}\n
      Number of sentences in 16th century: {len(century_16th)}\n
      Number of sentences in 18th century: {len(century_18th)}\n
      Number of sentences in 19th century: {len(century_19th)}\n
      Number of sentences in 20th century: {len(century_20th)}""")

Number of sentences in 5th century: 31

      Number of sentences in 12th century: 62

      Number of sentences in 13th century: 624

      Number of sentences in 14th century: 114

      Number of sentences in 15th century: 581

      Number of sentences in 16th century: 343

      Number of sentences in 18th century: 238

      Number of sentences in 19th century: 872

      Number of sentences in 20th century: 1529


In [11]:
class CleanCorpusIterator:
    def __init__(self, data):
        self.data = data
        # We pre-calculate this once so it's fast later
        self._length = sum(1 for item in self.data if isinstance(item, list) and item != "__DOC_END__")
        
    def __iter__(self):
        for item in self.data:
            if item == "__DOC_END__":
                continue
            if isinstance(item, list):
                yield item
                
    def __len__(self):
        # This fixes the TypeError
        return self._length

In [12]:
century_18th = CleanCorpusIterator(century_18th)

In [13]:
century_20th = CleanCorpusIterator(century_20th)

In [14]:
class LossLogger(CallbackAny2Vec):
    '''Callback to log loss after each epoch.'''
    def __init__(self):
        self.epoch = 0
        self.last_loss = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        # Calculate loss for the current epoch specifically
        current_loss = loss - self.last_loss
        self.last_loss = loss
        print(f"Epoch {self.epoch} Loss: {current_loss}")
        self.epoch += 1

In [15]:
BASE_MODEL_PATH = "Sin/cc.si.300.bin"
OUTPUT_DIR = "models/"
CENTURIES = ['13', '14', '15', '18', '19', '20']

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [16]:
def train_diachronic_models(base_model_path, century_data_dict):
    for c in CENTURIES:
        if c not in century_data_dict:
            continue
            
        print(f"\nProcessing Century: {c}")
        model = load_facebook_model(base_model_path)

        clean_corpus = [[w for w in s if len(w) > 1] for s in century_data_dict[c]]
        clean_corpus = [s for s in clean_corpus if s]
        
        # Identify words unique to this century's corpus
        century_vocab = set(word for sentence in clean_corpus for word in sentence)

        model.build_vocab(clean_corpus, update=True)

        model.train(
            corpus_iterable=clean_corpus,
            total_examples=len(clean_corpus),
            epochs=30,
            start_alpha=0.01,
            end_alpha=0.0001
        )

        # Vocabulary Pruning: Remove all words NOT in the current century corpus
        model.wv.fill_norms(force=True)
        
        # Get indices of words to keep
        keep_indices = [model.wv.key_to_index[w] for w in century_vocab if w in model.wv.key_to_index]
        model.wv.vectors = model.wv.vectors[keep_indices]
        
        # Re-map index and key lists
        new_index_to_key = [w for w in century_vocab if w in model.wv.key_to_index]
        model.wv.key_to_index = {w: i for i, w in enumerate(new_index_to_key)}
        model.wv.index_to_key = new_index_to_key

        save_path = os.path.join(OUTPUT_DIR, f"sinhala_ft_century_{c}.model")
        model.save(save_path)
        print(f"Saved pruned model ({len(model.wv.index_to_key)} words): {save_path}")

In [17]:
train_diachronic_models(BASE_MODEL_PATH, century_tokens_loaded)


Processing Century: 13
Saved pruned model (3516 words): models/sinhala_ft_century_13.model

Processing Century: 14
Saved pruned model (1917 words): models/sinhala_ft_century_14.model

Processing Century: 15
Saved pruned model (2768 words): models/sinhala_ft_century_15.model

Processing Century: 18
Saved pruned model (1409 words): models/sinhala_ft_century_18.model

Processing Century: 19
Saved pruned model (3979 words): models/sinhala_ft_century_19.model

Processing Century: 20
Saved pruned model (5367 words): models/sinhala_ft_century_20.model


In [18]:
base_model = load_facebook_model(BASE_MODEL_PATH)
m20 = FastText.load("models/sinhala_ft_century_20.model")

test_word = "අම්මා" 

if test_word in base_model.wv and test_word in m20.wv:
    v_base = base_model.wv[test_word].reshape(1, -1)
    v_13 = m20.wv[test_word].reshape(1, -1)
    
    similarity = cosine_similarity(v_base, v_13)[0][0]
    
    if similarity < 0.9999:
        print(f"✅ Success: Vector for '{test_word}' has moved. Similarity: {similarity:.4f}")
    else:
        print(f"❌ Warning: Vector for '{test_word}' is identical. Training might not have triggered.")

✅ Success: Vector for 'අම්මා' has moved. Similarity: 0.7281
